# Step 2: AWS S3 Bucket Setup and Verification

**Purpose**: Initialize and verify S3 bucket for the machine learning project

**What this notebook does**:
- Creates an S3 bucket if it doesn't exist
- Verifies bucket access and permissions
- Lists existing data files in the bucket

**Prerequisites**: Run `01_setup_env.ipynb` first to create `.env` configuration

## S3 Bucket Creation and Verification

This section handles the initial setup of your S3 storage for the project.

In [1]:
# Initialize AWS S3 client and load configuration
# This script creates the S3 bucket needed for storing ML artifacts
import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv
import os

# Load environment variables containing bucket name and region
load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')
region = os.getenv('AWS_DEFAULT_REGION')
s3 = boto3.client('s3')

# Create bucket if it doesn't exist (handles region-specific creation)
try:
    s3.head_bucket(Bucket=bucket_name)
except ClientError as e:
    if e.response['Error']['Code'] == '404':
        print(f"Bucket {bucket_name} doesn't exist. Creating...")
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
        print(f"Bucket {bucket_name} created successfully")
    else:
        raise

# Verify bucket contents - list any existing data files
response = s3.list_objects_v2(Bucket=bucket_name, Prefix='data/raw/')
for obj in response.get('Contents', []):
    print(f"Found: {obj['Key']} ({obj['Size']:,} bytes)")

Found: data/raw/ (0 bytes)
Found: data/raw/machines.csv (8,640,043 bytes)


## Next Steps

Once your S3 bucket is set up:

1. **Upload data**: Use the data preparation notebooks to upload training data
2. **Train models**: Run the model training pipeline
3. **Deploy**: Create endpoints for real-time inference

**Important**: This bucket will store:
- Raw training data (`data/raw/`)
- Processed datasets (`data/processed/`)
- Trained model artifacts (`models/`)
- Experiment results (`experiments/`)